# Vidarbha Backtest — Data Collection Pipeline

Extracts 5-year (Jun 2021 – Jun 2026) multi-source satellite and climate data for 4 Vidarbha sites.

**Sites:** Yavatmal, Akola, Wardha, Chandrapur

**Sources:**
- GEE: CHIRPS, ERA5-Land, Sentinel-2, Sentinel-1, MODIS NDVI composite, MODIS LST, SRTM DEM
- Bhoonidhi API: CartoSat-1 DEM (download + extract), LISS3 scene catalog, AWIFS_BOA scene catalog

**Output:** One `.xlsx` per site with 10 sheets.

**Backtest vs nowcast:** This notebook builds the historical baseline. The nowcast swaps CHIRPS → GPM IMERG Early, adds the AWIFS 15-day NDVI composite, and computes anomalies against the historical medians produced here.

---
**Bhoonidhi note:** Requires your public IP to be whitelisted at NRSC. Currently whitelisted: `202.94.161.105`. Bhoonidhi cells will skip gracefully and log a warning if the IP is blocked.

In [1]:
# ── Dependencies ──────────────────────────────────────────────────────────────
# Run once to install if needed:
#   pip install earthengine-api pandas openpyxl requests rasterio numpy scipy

import ee
import pandas as pd
import numpy as np
import requests
import datetime
import time
import os
import json
from pathlib import Path
from scipy.ndimage import sobel

try:
    import rasterio
    import rasterio.transform
    RASTERIO_OK = True
except ImportError:
    RASTERIO_OK = False
    print("WARNING: rasterio not installed — CartoSat DEM extraction will be skipped.")
    print("         Install with: pip install rasterio")

print("All imports OK.")

All imports OK.


In [2]:
# ── GEE Authentication ────────────────────────────────────────────────────────
GEE_PROJECT = 'earth-mrv'   # ← your GCP project ID

try:
    ee.Initialize(project=GEE_PROJECT)
    print("GEE initialised.")
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)
    print("GEE authenticated and initialised.")

Enter verification code:  4/1AXEQxICaeFcDqQLReK4Li4dKWm7-mD_FnT1qV_qvptihJPUWAKpWfHgOVKM



Successfully saved authorization token.
GEE authenticated and initialised.


In [3]:
# ── Bhoonidhi Auth + Token Manager ───────────────────────────────────────────
BHOONIDHI_BASE = 'https://bhoonidhi-api.nrsc.gov.in'
BHOONIDHI_USER = 'rakheja'
BHOONIDHI_PASS = 'Alskdj1029*'

class BhooToken:
    """Manages a Bhoonidhi JWT, auto-refreshing when <90 s remain."""
    def __init__(self):
        self._token = None
        self._refresh = None
        self._expiry = 0
        self._available = None   # None = untested, True/False after first attempt

    def _login(self):
        try:
            r = requests.post(
                f'{BHOONIDHI_BASE}/auth/token',
                json={'userId': BHOONIDHI_USER, 'password': BHOONIDHI_PASS, 'grant_type': 'password'},
                timeout=15
            )
            if r.status_code == 200:
                d = r.json()
                self._token   = d['access_token']
                self._refresh = d['refresh_token']
                self._expiry  = time.time() + d.get('expires_in', 1200) - 90
                self._available = True
                print("Bhoonidhi: authenticated.")
            else:
                self._available = False
                print(f"Bhoonidhi: auth failed (HTTP {r.status_code}) — ISRO cells will be skipped.")
        except Exception as e:
            self._available = False
            print(f"Bhoonidhi: unreachable ({e}) — ISRO cells will be skipped.")
            print("  (Is your IP whitelisted? Currently whitelisted: 202.94.161.105)")

    def _do_refresh(self):
        try:
            r = requests.post(
                f'{BHOONIDHI_BASE}/auth/token',
                json={'userId': BHOONIDHI_USER, 'refresh_token': self._refresh, 'grant_type': 'refresh_token'},
                timeout=15
            )
            if r.status_code == 200:
                d = r.json()
                self._token   = d['access_token']
                self._refresh = d['refresh_token']
                self._expiry  = time.time() + d.get('expires_in', 1200) - 90
            else:
                self._login()   # fall back to full login
        except Exception:
            self._login()

    @property
    def ok(self):
        if self._available is None:
            self._login()
        return self._available

    @property
    def token(self):
        if not self.ok:
            return None
        if time.time() > self._expiry:
            self._do_refresh()
        return self._token

    def headers(self):
        return {'Authorization': f'Bearer {self.token}', 'Content-Type': 'application/json'}


bhoo = BhooToken()
_ = bhoo.ok   # trigger the first login attempt now

Bhoonidhi: unreachable (HTTPSConnectionPool(host='bhoonidhi-api.nrsc.gov.in', port=443): Max retries exceeded with url: /auth/token (Caused by ConnectTimeoutError(<HTTPSConnection(host='bhoonidhi-api.nrsc.gov.in', port=443) at 0x129ffc410>, 'Connection to bhoonidhi-api.nrsc.gov.in timed out. (connect timeout=15)'))) — ISRO cells will be skipped.
  (Is your IP whitelisted? Currently whitelisted: 202.94.161.105)


In [4]:
# ── Locations & Time Window ───────────────────────────────────────────────────
LOCATIONS = {
    'Yavatmal': {
        'lat': 20.3890, 'lon': 78.1203,
        'buffer_m': 5000,
        'notes': 'Cotton distress epicentre, kharif cotton+soybean'
    },
    'Akola': {
        'lat': 20.7059, 'lon': 77.0082,
        'buffer_m': 5000,
        'notes': 'Cotton/soybean, western Vidarbha'
    },
    'Wardha': {
        'lat': 20.7453, 'lon': 78.6022,
        'buffer_m': 5000,
        'notes': 'Central cotton belt'
    },
    'Chandrapur': {
        'lat': 19.9615, 'lon': 79.2961,
        'buffer_m': 5000,
        'notes': 'South-east Vidarbha, rice + cotton contrast'
    },
}

START_DATE = '2021-06-01'
END_DATE   = '2026-06-01'

OUT_DIR = Path('./vidarbha_outputs')
OUT_DIR.mkdir(exist_ok=True)
DEM_DIR = OUT_DIR / 'dem_tiles'
DEM_DIR.mkdir(exist_ok=True)

print(f"Sites: {list(LOCATIONS.keys())}")
print(f"Window: {START_DATE} → {END_DATE}")
print(f"Output directory: {OUT_DIR.resolve()}")

Sites: ['Yavatmal', 'Akola', 'Wardha', 'Chandrapur']
Window: 2021-06-01 → 2026-06-01
Output directory: /Users/rakheja/Documents/GCL26/vidarbha_outputs


In [5]:
# ── GEE Helper: date-list for ERA5 daily aggregation ─────────────────────────
def make_date_list(start, end):
    s = datetime.datetime.strptime(start, '%Y-%m-%d')
    e = datetime.datetime.strptime(end,   '%Y-%m-%d')
    return [(s + datetime.timedelta(days=i)).strftime('%Y-%m-%d')
            for i in range((e - s).days)]

def geom_for(cfg):
    return ee.Geometry.Point([cfg['lon'], cfg['lat']]).buffer(cfg['buffer_m'])

print(f"Total days in window: {len(make_date_list(START_DATE, END_DATE))}")

Total days in window: 1826


In [6]:
# ── GEE Source 1: CHIRPS Daily Rainfall ──────────────────────────────────────
def get_chirps(geom, start, end):
    col = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
             .filterDate(start, end)
             .filterBounds(geom)
             .select('precipitation'))
    feats = col.map(lambda img: ee.Feature(None, {
        'date':          img.date().format('yyyy-MM-dd'),
        'precipitation': img.reduceRegion(ee.Reducer.mean(), geom, 5000).get('precipitation')
    })).getInfo()['features']
    df = pd.DataFrame([f['properties'] for f in feats])
    df['date'] = pd.to_datetime(df['date'])
    return df.sort_values('date').reset_index(drop=True)

print("CHIRPS extractor ready.")

CHIRPS extractor ready.


In [7]:
# ── GEE Source 2: ERA5-Land Daily Climate ─────────────────────────────────────
def get_era5(geom, start, end):
    era5_raw = (ee.ImageCollection('ECMWF/ERA5_LAND/HOURLY')
                  .filterDate(start, end)
                  .select(['temperature_2m', 'dewpoint_temperature_2m',
                           'volumetric_soil_water_layer_1',
                           'volumetric_soil_water_layer_2',
                           'volumetric_soil_water_layer_3',
                           'total_evaporation_hourly',
                           'surface_net_solar_radiation_hourly']))

    def preprocess(img):
        t   = img.select('temperature_2m').subtract(273.15).rename('temp_2m_C')
        td  = img.select('dewpoint_temperature_2m').subtract(273.15).rename('dewpoint_C')
        es  = t.expression('0.6108 * exp((17.27 * b()) / (b() + 237.3))')
        ea  = td.expression('0.6108 * exp((17.27 * b()) / (b() + 237.3))')
        vpd = es.subtract(ea).rename('VPD_kPa')
        sm1 = img.select('volumetric_soil_water_layer_1').rename('soil_moisture_l1')
        sm2 = img.select('volumetric_soil_water_layer_2').rename('soil_moisture_l2')
        sm3 = img.select('volumetric_soil_water_layer_3').rename('soil_moisture_l3')
        ev  = img.select('total_evaporation_hourly').multiply(-1000).rename('evaporation_mm')
        sr  = img.select('surface_net_solar_radiation_hourly').divide(1e6).rename('net_solar_rad_MJ')
        return img.addBands([t, vpd, sm1, sm2, sm3, ev, sr]).select(
            ['temp_2m_C', 'VPD_kPa', 'soil_moisture_l1',
             'soil_moisture_l2', 'soil_moisture_l3', 'evaporation_mm', 'net_solar_rad_MJ'])

    era5 = era5_raw.map(preprocess)
    dates = ee.List([ee.Date(start).advance(i, 'day').format('yyyy-MM-dd')
                     for i in range((datetime.datetime.strptime(end, '%Y-%m-%d') -
                                     datetime.datetime.strptime(start, '%Y-%m-%d')).days)])

    def daily_mean(d_str):
        d = ee.String(d_str)
        img = era5.filterDate(ee.Date(d), ee.Date(d).advance(1, 'day')).mean()
        s   = img.reduceRegion(ee.Reducer.mean(), geom, 9000)
        return ee.Feature(None, {
            'date': d, 'temp_2m_C': s.get('temp_2m_C'), 'VPD_kPa': s.get('VPD_kPa'),
            'soil_moisture_l1': s.get('soil_moisture_l1'),
            'soil_moisture_l2': s.get('soil_moisture_l2'),
            'soil_moisture_l3': s.get('soil_moisture_l3'),
            'evaporation_mm':   s.get('evaporation_mm'),
            'net_solar_rad_MJ': s.get('net_solar_rad_MJ')
        })

    feats = ee.FeatureCollection(dates.map(daily_mean)).getInfo()['features']
    df = pd.DataFrame([f['properties'] for f in feats])
    df['date'] = pd.to_datetime(df['date'])
    return df.sort_values('date').reset_index(drop=True)

print("ERA5 extractor ready.")

ERA5 extractor ready.


In [8]:
# ── GEE Source 3: Sentinel-2 Optical Indices ──────────────────────────────────
def get_sentinel2(geom, start, end):
    s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterDate(start, end)

    def mask_and_index(img):
        scl  = img.select('SCL')
        mask = scl.eq(4).Or(scl.eq(5)).Or(scl.eq(6)).Or(scl.eq(7))
        sc   = img.updateMask(mask).divide(10000.0)
        ndvi = sc.normalizedDifference(['B8', 'B4']).rename('NDVI')
        ndwi = sc.normalizedDifference(['B8', 'B11']).rename('NDWI')
        ndre = sc.normalizedDifference(['B8', 'B5']).rename('NDRE')
        mndwi= sc.normalizedDifference(['B3', 'B11']).rename('MNDWI')
        savi = sc.expression('((B8-B4)/(B8+B4+0.5))*1.5',
                             {'B8': sc.select('B8'), 'B4': sc.select('B4')}).rename('SAVI')
        evi  = sc.expression('2.5*((B8-B4)/(B8+6*B4-7.5*B2+1))',
                             {'B8': sc.select('B8'), 'B4': sc.select('B4'),
                              'B2': sc.select('B2')}).rename('EVI')
        return img.addBands([ndvi, ndwi, ndre, mndwi, savi, evi,
                              sc.select('B11').rename('SWIR1'),
                              sc.select('B12').rename('SWIR2')])

    s2p = s2.map(mask_and_index).select(['NDVI','NDWI','NDRE','MNDWI','SAVI','EVI','SWIR1','SWIR2'])
    median_ndvi = s2p.select('NDVI').median()
    s2f = s2p.map(lambda img: img.addBands(
        img.select('NDVI').subtract(median_ndvi).rename('NDVI_Anomaly')))

    def extract(img):
        r = {b: img.reduceRegion(ee.Reducer.mean(), geom, 100).get(b)
             for b in ['NDVI','NDWI','NDRE','MNDWI','SAVI','EVI','SWIR1','SWIR2','NDVI_Anomaly']}
        r['date'] = img.date().format('yyyy-MM-dd')
        return ee.Feature(None, r)

    feats = s2f.filterBounds(geom).map(extract).getInfo()['features']
    df = pd.DataFrame([f['properties'] for f in feats])
    if df.empty:
        return df
    df['date'] = pd.to_datetime(df['date'])
    return df.groupby('date').mean().reset_index().sort_values('date').reset_index(drop=True)

print("Sentinel-2 extractor ready.")

Sentinel-2 extractor ready.


In [9]:
# ── GEE Source 4: Sentinel-1 SAR ──────────────────────────────────────────────
def get_sentinel1(geom, start, end):
    s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
            .filterBounds(geom).filterDate(start, end)
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
            .filter(ee.Filter.eq('instrumentMode', 'IW')))

    def extract(img):
        s = img.reduceRegion(ee.Reducer.mean(), geom, 20)
        return ee.Feature(None, {
            'date':  img.date().format('yyyy-MM-dd'),
            'VV_dB': s.get('VV'),
            'VH_dB': s.get('VH')
        })

    feats = s1.map(extract).getInfo()['features']
    df = pd.DataFrame([f['properties'] for f in feats])
    if df.empty:
        return df
    df['date'] = pd.to_datetime(df['date'])
    return df.groupby('date').mean().reset_index().sort_values('date').reset_index(drop=True)

print("Sentinel-1 extractor ready.")

Sentinel-1 extractor ready.


In [10]:
# ── GEE Source 5: MODIS MOD13Q1 — 16-day NDVI/EVI composite ──────────────────
# Cloud-free composites; fills Sentinel-2 gaps during monsoon cloud cover.
# Scale factor: 0.0001 (raw values are scaled integers)
def get_modis_ndvi(geom, start, end):
    col = (ee.ImageCollection('MODIS/061/MOD13Q1')
             .filterDate(start, end)
             .filterBounds(geom)
             .select(['NDVI', 'EVI']))

    def extract(img):
        scaled = img.multiply(0.0001)
        s = scaled.reduceRegion(ee.Reducer.mean(), geom, 250)
        return ee.Feature(None, {
            'date':       img.date().format('yyyy-MM-dd'),
            'NDVI_modis': s.get('NDVI'),
            'EVI_modis':  s.get('EVI')
        })

    feats = col.map(extract).getInfo()['features']
    df = pd.DataFrame([f['properties'] for f in feats])
    if df.empty:
        return df
    df['date'] = pd.to_datetime(df['date'])
    return df.sort_values('date').reset_index(drop=True)

print("MODIS NDVI/EVI extractor ready.")

MODIS NDVI/EVI extractor ready.


In [11]:
# ── GEE Source 6: MODIS MOD11A2 — 8-day Land Surface Temperature ─────────────
# LST is the skin temperature of the land surface — more directly relevant to
# crop heat stress than ERA5 2m air temperature (which is modelled, not observed).
# Scale: raw × 0.02 − 273.15 → Celsius
def get_modis_lst(geom, start, end):
    col = (ee.ImageCollection('MODIS/061/MOD11A2')
             .filterDate(start, end)
             .filterBounds(geom)
             .select(['LST_Day_1km', 'LST_Night_1km']))

    def extract(img):
        # Apply scale + offset: K → Celsius
        scaled = img.multiply(0.02).subtract(273.15)
        s = scaled.reduceRegion(ee.Reducer.mean(), geom, 1000)
        return ee.Feature(None, {
            'date':         img.date().format('yyyy-MM-dd'),
            'LST_Day_C':    s.get('LST_Day_1km'),
            'LST_Night_C':  s.get('LST_Night_1km')
        })

    feats = col.map(extract).getInfo()['features']
    df = pd.DataFrame([f['properties'] for f in feats])
    if df.empty:
        return df
    df['date'] = pd.to_datetime(df['date'])
    return df.sort_values('date').reset_index(drop=True)

print("MODIS LST extractor ready.")

MODIS LST extractor ready.


In [12]:
# ── GEE Source 7: SRTM DEM — static elevation + slope ────────────────────────
# Returns a single-row dict per site: elevation (m), slope (degrees), aspect (degrees)
def get_srtm_static(geom):
    srtm  = ee.Image('USGS/SRTMGL1_003')
    slope = ee.Terrain.slope(srtm)
    aspect= ee.Terrain.aspect(srtm)

    elev = srtm.select('elevation').reduceRegion(ee.Reducer.mean(), geom, 30).getInfo().get('elevation')
    slp  = slope.select('slope').reduceRegion(ee.Reducer.mean(), geom, 30).getInfo().get('slope')
    asp  = aspect.select('aspect').reduceRegion(ee.Reducer.mean(), geom, 30).getInfo().get('aspect')

    return {'source': 'SRTM_v3_30m', 'elevation_m': elev, 'slope_deg': slp, 'aspect_deg': asp}

print("SRTM static extractor ready.")

SRTM static extractor ready.


In [13]:
# ── Bhoonidhi Source 1: CartoSat-1 DEM — download + extract ──────────────────
# Tiles are 1°×1° GeoTIFFs at 30m. We download only the 3 unique tiles needed.
# Elevation + slope extracted at each site coordinate.

def cartosat_tile_id(lat, lon):
    """Return Bhoonidhi tile ID for the 1°x1° tile containing (lat, lon)."""
    lat_floor = int(np.floor(lat))
    lon_floor = int(np.floor(lon))
    return f'P5_PAN_CD_N{lat_floor:02d}_000_E{lon_floor:03d}_000_30m'

def download_cartosat_tile(tile_id, out_path):
    """Download a CartoSat DEM tile from Bhoonidhi if not already present."""
    if out_path.exists():
        print(f"  DEM tile {tile_id}: already downloaded.")
        return True
    if not bhoo.ok:
        print(f"  DEM tile {tile_id}: Bhoonidhi unavailable, skipping.")
        return False
    url = f"{BHOONIDHI_BASE}/download?id={tile_id}&collection=CartoSat-1_PAN_CartoDEM_30m"
    print(f"  Downloading DEM tile {tile_id} …", end=' ', flush=True)
    try:
        r = requests.get(url, headers=bhoo.headers(), stream=True, timeout=300)
        if r.status_code == 200:
            with open(out_path, 'wb') as f:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)
            print(f"done ({out_path.stat().st_size / 1e6:.1f} MB).")
            return True
        else:
            print(f"failed (HTTP {r.status_code}).")
            return False
    except Exception as e:
        print(f"error: {e}")
        return False

def extract_cartosat_point(tile_path, lat, lon):
    """Extract elevation + slope at (lat, lon) from a downloaded DEM GeoTIFF."""
    if not RASTERIO_OK:
        return {'elevation_cartosat_m': None, 'slope_cartosat_deg': None}
    with rasterio.open(tile_path) as src:
        row, col = rasterio.transform.rowcol(src.transform, lon, lat)
        dem = src.read(1).astype(float)
        elev = float(dem[row, col])
        # Approximate metres-per-pixel
        px_deg = abs(src.transform.a)
        px_m   = px_deg * 111320 * np.cos(np.radians(lat))
        dx = sobel(dem, axis=1) / (8 * px_m)
        dy = sobel(dem, axis=0) / (8 * px_m)
        slope_arr = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))
        slope = float(slope_arr[row, col])
    return {'elevation_cartosat_m': round(elev, 1), 'slope_cartosat_deg': round(slope, 3)}

def get_cartosat_for_site(lat, lon):
    tile_id   = cartosat_tile_id(lat, lon)
    tile_path = DEM_DIR / f"{tile_id}.tif"
    ok = download_cartosat_tile(tile_id, tile_path)
    if ok and tile_path.exists():
        result = extract_cartosat_point(tile_path, lat, lon)
    else:
        result = {'elevation_cartosat_m': None, 'slope_cartosat_deg': None}
    result['tile_id'] = tile_id
    result['source']  = 'CartoSat-1_PAN_CartoDEM_30m'
    return result

print("CartoSat DEM extractor ready.")

CartoSat DEM extractor ready.


In [14]:
# ── Bhoonidhi Source 2: LISS3 Scene Catalog ───────────────────────────────────
# Searches both ResourceSat-2 and ResourceSat-2A LISS3 for scenes that
# intersect the site point. Uses 6-month windows to stay under the API limit.
# Returns metadata only (no download). Online=Y scenes are directly downloadable.

LISS3_COLLECTIONS = ['ResourceSat-2_LISS3_L2', 'ResourceSat-2A_LISS3_L2']

def search_bhoonidhi_scenes(collection, lat, lon, window_start, window_end, limit=500):
    """Search Bhoonidhi for scenes in a ≤6-month window at a point."""
    if not bhoo.ok:
        return []
    payload = {
        'collections': [collection],
        'intersects':  {'type': 'Point', 'coordinates': [lon, lat]},
        'datetime':    f"{window_start}T00:00:00Z/{window_end}T23:59:59Z",
        'limit':       limit
    }
    try:
        r = requests.post(f'{BHOONIDHI_BASE}/data/search',
                          headers=bhoo.headers(), json=payload, timeout=30)
        time.sleep(0.4)   # 3 req/s rate limit
        if r.status_code != 200:
            return []
        d = r.json()
        if d.get('ErrorCode'):
            return []
        return d.get('features', [])
    except Exception:
        return []

def make_6month_windows(start_str, end_str):
    """Split a date range into ≤6-month windows."""
    windows = []
    cur = datetime.datetime.strptime(start_str, '%Y-%m-%d')
    end = datetime.datetime.strptime(end_str,   '%Y-%m-%d')
    while cur < end:
        nxt = min(cur + datetime.timedelta(days=180), end)
        windows.append((cur.strftime('%Y-%m-%d'), nxt.strftime('%Y-%m-%d')))
        cur = nxt
    return windows

def get_liss3_catalog(lat, lon, start, end):
    records = []
    windows = make_6month_windows(start, end)
    for col in LISS3_COLLECTIONS:
        for ws, we in windows:
            feats = search_bhoonidhi_scenes(col, lat, lon, ws, we)
            for f in feats:
                p = f.get('properties', {})
                records.append({
                    'collection':    col,
                    'scene_id':      f.get('id', ''),
                    'date':          p.get('datetime', '')[:10],
                    'online':        p.get('Online', ''),
                    'path':          p.get('Path', ''),
                    'row':           p.get('Row', ''),
                    'platform':      p.get('Platform', ''),
                    'instrument':    p.get('Instrument', ''),
                    'centre_lat':    p.get('CentreLatLon', [None, None])[0],
                    'centre_lon':    p.get('CentreLatLon', [None, None])[1],
                    'item_status':   p.get('ItemStatus', '')
                })
    if not records:
        return pd.DataFrame()
    df = pd.DataFrame(records)
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    return df.drop_duplicates('scene_id').sort_values('date').reset_index(drop=True)

print("LISS3 catalog extractor ready.")

LISS3 catalog extractor ready.


In [15]:
# ── Bhoonidhi Source 3: AWIFS_BOA Scene Catalog ───────────────────────────────
# AWIFS_BOA is the PMFBY-aligned sensor (56m, atmospherically corrected).
# Coverage: 2021=3 scenes, 2022=2, 2023=30, 2024=386, 2025=346.
# Metadata only — Online=Y scenes are downloadable for production integration.

AWIFS_COLLECTIONS = ['ResourceSat-2_AWIFS_BOA', 'ResourceSat-2A_AWIFS_BOA']

def get_awifs_catalog(lat, lon, start, end):
    records = []
    windows = make_6month_windows(start, end)
    for col in AWIFS_COLLECTIONS:
        for ws, we in windows:
            feats = search_bhoonidhi_scenes(col, lat, lon, ws, we)
            for f in feats:
                p = f.get('properties', {})
                records.append({
                    'collection':  col,
                    'scene_id':    f.get('id', ''),
                    'date':        p.get('datetime', '')[:10],
                    'online':      p.get('Online', ''),
                    'path':        p.get('Path', ''),
                    'row':         p.get('Row', ''),
                    'platform':    p.get('Platform', ''),
                    'instrument':  p.get('Instrument', ''),
                    'coverage':    p.get('Coverage', ''),
                    'item_status': p.get('ItemStatus', '')
                })
    if not records:
        return pd.DataFrame()
    df = pd.DataFrame(records)
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    return df.drop_duplicates('scene_id').sort_values('date').reset_index(drop=True)

print("AWIFS catalog extractor ready.")

AWIFS catalog extractor ready.


In [16]:
# ── Main Pipeline — loop over all 4 sites ─────────────────────────────────────
# Each site produces one Excel workbook with 10 sheets.
# Runtime estimate: ~30-60 min/site for GEE (ERA5 dominates); ~5 min/site for Bhoonidhi.

for site, cfg in LOCATIONS.items():
    print(f"\n{'='*60}")
    print(f"  Processing: {site}  ({cfg['lat']:.4f}N, {cfg['lon']:.4f}E)")
    print(f"  {cfg['notes']}")
    print(f"{'='*60}")

    geom   = geom_for(cfg)
    outfile = OUT_DIR / f"{site.lower()}_backtest_all_sources.xlsx"

    # ── GEE extractions ────────────────────────────────────────────────────────
    print("[1/9] CHIRPS rainfall …")
    df_chirps = get_chirps(geom, START_DATE, END_DATE)
    print(f"      {len(df_chirps)} daily records")

    print("[2/9] ERA5-Land climate (slow — ~20 min) …")
    df_era5 = get_era5(geom, START_DATE, END_DATE)
    print(f"      {len(df_era5)} daily records")

    print("[3/9] Sentinel-2 optical indices …")
    df_s2 = get_sentinel2(geom, START_DATE, END_DATE)
    print(f"      {len(df_s2)} clear-sky scenes")

    print("[4/9] Sentinel-1 SAR …")
    df_s1 = get_sentinel1(geom, START_DATE, END_DATE)
    print(f"      {len(df_s1)} SAR acquisitions")

    print("[5/9] MODIS 16-day NDVI/EVI composite …")
    df_modis_ndvi = get_modis_ndvi(geom, START_DATE, END_DATE)
    print(f"      {len(df_modis_ndvi)} composite periods")

    print("[6/9] MODIS 8-day Land Surface Temperature …")
    df_modis_lst = get_modis_lst(geom, START_DATE, END_DATE)
    print(f"      {len(df_modis_lst)} 8-day LST observations")

    print("[7/9] SRTM DEM (static) …")
    srtm_static = get_srtm_static(geom)
    srtm_static['site'] = site
    print(f"      elevation={srtm_static['elevation_m']:.1f}m  slope={srtm_static['slope_deg']:.2f}°")

    # ── Bhoonidhi extractions ──────────────────────────────────────────────────
    print("[8/9] CartoSat-1 DEM download + extraction (Bhoonidhi) …")
    cartosat = get_cartosat_for_site(cfg['lat'], cfg['lon'])
    cartosat['site'] = site
    print(f"      elevation={cartosat['elevation_cartosat_m']}m  "
          f"slope={cartosat['slope_cartosat_deg']}°  tile={cartosat['tile_id']}")

    # Merge static elevation sources into a single sheet
    df_elevation = pd.DataFrame([{
        'site':                    site,
        'lat':                     cfg['lat'],
        'lon':                     cfg['lon'],
        'elevation_srtm_m':        srtm_static.get('elevation_m'),
        'slope_srtm_deg':          srtm_static.get('slope_deg'),
        'aspect_srtm_deg':         srtm_static.get('aspect_deg'),
        'elevation_cartosat_m':    cartosat.get('elevation_cartosat_m'),
        'slope_cartosat_deg':      cartosat.get('slope_cartosat_deg'),
        'cartosat_tile':           cartosat.get('tile_id'),
    }])

    print("[9/9] Bhoonidhi ISRO scene catalogs (LISS3 + AWIFS) …")
    print("      Searching LISS3 …")
    df_liss3 = get_liss3_catalog(cfg['lat'], cfg['lon'], START_DATE, END_DATE)
    n_online_liss3 = len(df_liss3[df_liss3['online'] == 'Y']) if not df_liss3.empty else 0
    print(f"      LISS3: {len(df_liss3)} scenes total, {n_online_liss3} Online=Y")

    print("      Searching AWIFS_BOA …")
    df_awifs = get_awifs_catalog(cfg['lat'], cfg['lon'], START_DATE, END_DATE)
    n_online_awifs = len(df_awifs[df_awifs['online'] == 'Y']) if not df_awifs.empty else 0
    print(f"      AWIFS: {len(df_awifs)} scenes total, {n_online_awifs} Online=Y")

    # ── Write Excel ────────────────────────────────────────────────────────────
    print(f"\nWriting → {outfile.name} …")
    with pd.ExcelWriter(outfile, engine='openpyxl') as writer:
        df_chirps.to_excel(writer, sheet_name='CHIRPS_Rainfall',      index=False)
        df_era5.to_excel(  writer, sheet_name='ERA5_Climate',         index=False)
        df_s2.to_excel(    writer, sheet_name='Sentinel2_Optical',    index=False)
        df_s1.to_excel(    writer, sheet_name='Sentinel1_SAR',        index=False)
        df_modis_ndvi.to_excel(writer, sheet_name='MODIS_NDVI_EVI',   index=False)
        df_modis_lst.to_excel( writer, sheet_name='MODIS_LST',        index=False)
        df_elevation.to_excel( writer, sheet_name='Elevation_Static', index=False)
        if not df_liss3.empty:
            df_liss3.to_excel(writer, sheet_name='LISS3_SceneCatalog', index=False)
        if not df_awifs.empty:
            df_awifs.to_excel(writer, sheet_name='AWIFS_SceneCatalog', index=False)

    print(f"SUCCESS: {outfile.name} written.")

print("\n=== Pipeline complete. ===")


  Processing: Yavatmal  (20.3890N, 78.1203E)
  Cotton distress epicentre, kharif cotton+soybean
[1/9] CHIRPS rainfall …
      1826 daily records
[2/9] ERA5-Land climate (slow — ~20 min) …
      1826 daily records
[3/9] Sentinel-2 optical indices …
      381 clear-sky scenes
[4/9] Sentinel-1 SAR …
      281 SAR acquisitions
[5/9] MODIS 16-day NDVI/EVI composite …
      115 composite periods
[6/9] MODIS 8-day Land Surface Temperature …
      230 8-day LST observations
[7/9] SRTM DEM (static) …
      elevation=435.0m  slope=3.49°
[8/9] CartoSat-1 DEM download + extraction (Bhoonidhi) …
  DEM tile P5_PAN_CD_N20_000_E078_000_30m: Bhoonidhi unavailable, skipping.
      elevation=Nonem  slope=None°  tile=P5_PAN_CD_N20_000_E078_000_30m
[9/9] Bhoonidhi ISRO scene catalogs (LISS3 + AWIFS) …
      Searching LISS3 …
      LISS3: 0 scenes total, 0 Online=Y
      Searching AWIFS_BOA …
      AWIFS: 0 scenes total, 0 Online=Y

Writing → yavatmal_backtest_all_sources.xlsx …
SUCCESS: yavatmal_backtest_

## Output sheets per site

| Sheet | Source | Cadence | Key columns |
|---|---|---|---|
| `CHIRPS_Rainfall` | GEE / CHIRPS | Daily | date, precipitation |
| `ERA5_Climate` | GEE / ERA5-Land | Daily | date, temp_2m_C, VPD_kPa, soil_moisture_l1/l2/l3, evaporation_mm, net_solar_rad_MJ |
| `Sentinel2_Optical` | GEE / Copernicus | Per clear scene ~5d | date, NDVI, NDWI, NDRE, MNDWI, SAVI, EVI, SWIR1/2, NDVI_Anomaly |
| `Sentinel1_SAR` | GEE / Copernicus | 6–12d | date, VV_dB, VH_dB |
| `MODIS_NDVI_EVI` | GEE / MODIS MOD13Q1 | 16-day composite | date, NDVI_modis, EVI_modis |
| `MODIS_LST` | GEE / MODIS MOD11A2 | 8-day | date, LST_Day_C, LST_Night_C |
| `Elevation_Static` | GEE SRTM + Bhoonidhi CartoSat | Static | elevation/slope from both sources |
| `LISS3_SceneCatalog` | Bhoonidhi ISRO | Per scene | scene_id, date, online, platform, path/row |
| `AWIFS_SceneCatalog` | Bhoonidhi ISRO | Per scene | scene_id, date, online, platform |

## What differs in the nowcast notebook

- Replace `CHIRPS` → `GPM IMERG Early` (6-hour lag instead of 5-day)
- Add `AWIFS 15-day NDVI composite` from Bhoonidhi (`ResourceSat-2_AWIFS_NDVI-10deg_15day_100m`) — the cloud-free current NDVI input to the trigger
- Filter all sources to the last 30–60 days (rolling window)
- Compute anomaly: `current_value − historical_median_for_this_calendar_week` using the 5-year baseline from this notebook